## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers.core import Dense,Dropout,Activation,Flatten
from keras.layers.convolutional import Conv2D,MaxPooling2D
from keras.utils import np_utils
from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

Using TensorFlow backend.


In [2]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default
#tf.enable_eager_execution()

### Collect Data

In [3]:
import pandas as pd

In [4]:
df= pd.read_csv('prices.csv')

### Check all columns in the dataset

In [5]:
df.shape

(851264, 7)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851264 entries, 0 to 851263
Data columns (total 7 columns):
date      851264 non-null object
symbol    851264 non-null object
open      851264 non-null float64
close     851264 non-null float64
low       851264 non-null float64
high      851264 non-null float64
volume    851264 non-null float64
dtypes: float64(5), object(2)
memory usage: 45.5+ MB


### Drop columns `date` and  `symbol`

In [7]:
df = df.drop(['date','symbol'],axis=1)

In [8]:
df.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [9]:
df = df.iloc[0:1000, :]

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
open      1000 non-null float64
close     1000 non-null float64
low       1000 non-null float64
high      1000 non-null float64
volume    1000 non-null float64
dtypes: float64(5)
memory usage: 39.1 KB


In [11]:
df['volume'] = df['volume']/1000000

In [12]:
df.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086


### Divide the data into train and test sets

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X = df.drop("volume", axis=1)
y = df["volume"]
test_size = 0.30 # taking 70:30 training and test set
seed = 7  # Random numbmer seeding for reapeatability of the code
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)

In [17]:
y_test.sample(n=300)

513     9.6167
222     0.8190
253     1.7017
983     0.7496
361     1.6807
541     4.0721
675     3.8972
992     5.2061
776     3.2766
283     2.7505
172     0.6411
621     0.8668
509     6.0671
886     2.1434
590     1.1697
220     0.8184
786    66.8617
482     2.7281
109     0.5480
242     0.9376
705     4.7414
264     1.2993
284     7.5999
270     7.7509
271     1.2282
422     1.8738
626     0.2947
597     0.5465
696    39.3357
906    18.0609
        ...   
725     7.1088
865     2.3448
148     1.8652
790     5.6972
166     0.4645
362    14.6768
46      0.5902
808    17.7465
893     2.3631
530     6.1094
693     8.1710
471     3.6370
644     3.4732
360     5.7652
558    11.9724
147     0.7667
158     1.4148
779     7.1893
616    15.7436
794     2.1160
50      0.8093
9       1.5235
925    12.4240
389    22.5116
800    28.6926
743     5.3421
836     3.0074
454     4.2787
296     1.3012
775     1.1222
Name: volume, Length: 300, dtype: float64

#### Convert Training and Test Data to numpy float32 arrays


In [18]:
X_train = np.array(X_train).astype(np.float32)

In [19]:
X_test = np.array(X_test).astype(np.float32)

In [20]:
y_train = np.array(y_train).astype(np.float32)

In [21]:
y_test = np.array(y_test).astype(np.float32)

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [22]:
from sklearn.preprocessing import Normalizer

In [23]:
X_train = Normalizer().fit_transform(X_train)

In [24]:
X_train

array([[0.50021225, 0.5003598 , 0.49549326, 0.503899  ],
       [0.5023256 , 0.50103617, 0.4931476 , 0.5034255 ],
       [0.5004301 , 0.49899533, 0.4962851 , 0.5042563 ],
       ...,
       [0.5001929 , 0.49977148, 0.4976645 , 0.50236005],
       [0.50253594, 0.49852157, 0.496339  , 0.50257486],
       [0.50121456, 0.4987676 , 0.49595362, 0.5040285 ]], dtype=float32)

In [25]:
X_test=Normalizer().fit_transform(X_test)

In [26]:
X_test

array([[0.5063693 , 0.49396917, 0.4904527 , 0.50896037],
       [0.49942613, 0.50043815, 0.49706477, 0.5030525 ],
       [0.4963037 , 0.50399053, 0.49479976, 0.50482607],
       ...,
       [0.49629417, 0.50301725, 0.49192008, 0.5086063 ],
       [0.49930164, 0.5008185 , 0.49753195, 0.50233537],
       [0.49550715, 0.50393987, 0.49483255, 0.50562644]], dtype=float32)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [27]:
tf.reset_default_graph()

In [28]:
#Input features
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')

#Normalize the data
x_n = tf.nn.l2_normalize(x,1)

#Actual Prices
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')

In [29]:
#W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

Instructions for updating:
Colocations handled automatically by placer.


2.Define a function to calculate prediction

In [30]:
y = tf.add(tf.matmul(x_n,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [31]:
loss = tf.reduce_mean(tf.square(y-y_),name='Loss')

In [32]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [34]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [35]:
for epoch in range(training_epochs):
        #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:X_train, y_:y_train})
    
    if epoch % 1 == 0:
        print ('Training loss at every iteration: ', epoch, ' is ', train_loss)

Training loss at every iteration:  0  is  305.42902
Training loss at every iteration:  1  is  298.32843
Training loss at every iteration:  2  is  292.82974
Training loss at every iteration:  3  is  288.57153
Training loss at every iteration:  4  is  285.2738
Training loss at every iteration:  5  is  282.72012
Training loss at every iteration:  6  is  280.74277
Training loss at every iteration:  7  is  279.21103
Training loss at every iteration:  8  is  278.02518
Training loss at every iteration:  9  is  277.10678
Training loss at every iteration:  10  is  276.39557
Training loss at every iteration:  11  is  275.8448
Training loss at every iteration:  12  is  275.41818
Training loss at every iteration:  13  is  275.08798
Training loss at every iteration:  14  is  274.8322
Training loss at every iteration:  15  is  274.634
Training loss at every iteration:  16  is  274.4808
Training loss at every iteration:  17  is  274.3619
Training loss at every iteration:  18  is  274.2699
Training lo

In [36]:
sess.close()

In [37]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

In [38]:
for epoch in range(training_epochs):
        #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:X_train, y_:y_train})
    
    if epoch % 5 == 0:
        print ('Training loss at every iteration: ', epoch, ' is ', train_loss)
        #numpy_array = W.eval(session=sess)

Training loss at every iteration:  0  is  305.42902
Training loss at every iteration:  5  is  282.72012
Training loss at every iteration:  10  is  276.39557
Training loss at every iteration:  15  is  274.634
Training loss at every iteration:  20  is  274.14355
Training loss at every iteration:  25  is  274.0067
Training loss at every iteration:  30  is  273.96872
Training loss at every iteration:  35  is  273.95813
Training loss at every iteration:  40  is  273.95538
Training loss at every iteration:  45  is  273.95447
Training loss at every iteration:  50  is  273.95425
Training loss at every iteration:  55  is  273.95407
Training loss at every iteration:  60  is  273.9541
Training loss at every iteration:  65  is  273.95413
Training loss at every iteration:  70  is  273.95416
Training loss at every iteration:  75  is  273.9542
Training loss at every iteration:  80  is  273.95425
Training loss at every iteration:  85  is  273.95428
Training loss at every iteration:  90  is  273.95425


In [39]:
Weight_array = W.eval(session=sess)
bias_array = b.eval(session=sess)

In [40]:
sess.close()

### Get the shapes and values of W and b

In [41]:
print('Weights Values =',Weight_array)
print('Bias Value =',bias_array)

Weights Values = [[1.4007461]
 [1.4055865]
 [1.3863369]
 [1.4173869]]
Bias Value = [2.8051822]


### Model Prediction on 1st Examples in Test Dataset

In [42]:
ypred = tf.add(tf.matmul(X_test,Weight_array),bias_array,name='output')

In [43]:
#Lets start graph Execution
sess1 = tf.Session()
ypred.eval(session=sess1)

array([[5.61012  ],
       [5.61028  ],
       [5.6102734],
       [5.6102104],
       [5.610093 ],
       [5.610279 ],
       [5.610261 ],
       [5.6102896],
       [5.6102657],
       [5.610283 ],
       [5.6102867],
       [5.6102858],
       [5.6102962],
       [5.6102734],
       [5.610194 ],
       [5.6102886],
       [5.610196 ],
       [5.610256 ],
       [5.609956 ],
       [5.6102614],
       [5.610283 ],
       [5.6102786],
       [5.610194 ],
       [5.61026  ],
       [5.610219 ],
       [5.61022  ],
       [5.6099815],
       [5.6102676],
       [5.6098657],
       [5.609933 ],
       [5.610265 ],
       [5.610116 ],
       [5.6102824],
       [5.6102037],
       [5.6102133],
       [5.6102877],
       [5.610281 ],
       [5.6102667],
       [5.610269 ],
       [5.6102657],
       [5.610289 ],
       [5.61026  ],
       [5.6102734],
       [5.610262 ],
       [5.610238 ],
       [5.610199 ],
       [5.6102457],
       [5.6102514],
       [5.610199 ],
       [5.610281 ],


## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [47]:
iris= pd.read_csv('Iris.csv')

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [48]:
iris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [51]:
X = iris.iloc[:,1:5].values
y = iris.iloc[:,5].values

from sklearn.preprocessing import LabelEncoder
encoder =  LabelEncoder()
y1 = encoder.fit_transform(y)

Y = pd.get_dummies(y1).values

### Splitting the data into feature set and target set

In [56]:
from sklearn.model_selection import train_test_split
X_train,X_test, y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=0)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [57]:
model = Sequential()
model.add(Dense(12, input_dim=4, activation='relu'))
model.add(Dense(3, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

### Model Training 

In [ ]:
model.fit(X_train, y_train, epochs=100)

### Model Prediction

In [63]:
y_predict = model.predict(X_test)
y_predict[0]

array([0.02129836, 0.1537929 , 0.82490873], dtype=float32)

### Save the Model

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?